In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (DistilBertTokenizer,DistilBertForSequenceClassification,Trainer,TrainingArguments)
from datasets import Dataset
df = pd.read_csv('/content/combined_dataset.csv')
df=df[['text','label']].dropna()
df['text']=df['text'].astype(str)
df['label']=df['label'].astype(int)
print("✅ Dataset loaded successfully!")
print(f"Total records after cleaning: {len(df)}")
print(df.head())
train_texts,val_texts,train_labels,val_labels=train_test_split(df['text'].tolist(),df['label'].tolist(),test_size=0.2,random_state=42)
tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
train_enc=tokenizer(train_texts,truncation=True,padding=True,max_length=512)
val_enc=tokenizer(val_texts,truncation=True,padding=True,max_length=512)
train_dataset=Dataset.from_dict({'input_ids': train_enc['input_ids'],'attention_mask': train_enc['attention_mask'],'labels':train_labels})
val_dataset=Dataset.from_dict({'input_ids':val_enc['input_ids'],'attention_mask':val_enc['attention_mask'],'labels':val_labels})
model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)
training_args=TrainingArguments(output_dir='/content/distilbert_model',eval_strategy='epoch',save_strategy='epoch',logging_strategy='epoch',per_device_train_batch_size=8,per_device_eval_batch_size=8,num_train_epochs=1,learning_rate=5e-5,weight_decay=0.01,load_best_model_at_end=True)
trainer=Trainer(model=model,args=training_args,train_dataset=train_dataset,eval_dataset=val_dataset)
trainer.train()
save_path="/content/Model_DistilBERT"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved successfully at: {save_path}")
metrics=trainer.evaluate()
print("\n📊 Evaluation Results:")
print(metrics)

✅ Dataset loaded successfully!
Total records after cleaning: 44898
                                                text  label
0  LONDON (Reuters) - Britain launched a fund on ...      1
1   You re carrying Mexican flags while chanting,...      0
2  Too bad for the gay pastor that Whole Foods ha...      0
3  Double standards everywhere!  Socialist Bernie...      0
4  MOSCOW (Reuters) - The Kremlin said on Wednesd...      1


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.006300,0.001347


✅ Model saved successfully at: /content/Model_DistilBERT



📊 Evaluation Results:
{'eval_loss': 0.0013469154946506023, 'eval_runtime': 129.3522, 'eval_samples_per_second': 69.423, 'eval_steps_per_second': 8.682, 'epoch': 1.0}
